In [42]:
from gurobipy import Model, GRB, quicksum
import numpy as np


class PSC_Optimizer:
    
    def __init__(self, task_times, deadlines, task_names):
        self.max_time = max(deadlines)  # Maximum available time
        self.task_times = task_times  # Array (length num tasks)
        self.deadlines = deadlines  # Array (length num tasks)
        self.task_names = task_names
        self.num_tasks = len(task_times)

    def OptimizeCalendar(self):
        if len(self.task_times) != len(self.deadlines):
            raise ValueError("Mismatch in task_times and deadlines length")

        # Create a new model
        model = Model("PSC-MIP-V2")

        # Decision Variables
        task_to_block = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="x")  # Task assigned to time block
        task_starting_time = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="t")  # Task start indicator

        
        # Define dicts for task times and deadlines
        task_times = {j: self.task_times[j] for j in range(self.num_tasks)}  # All tasks take task_times[j] time blocks
        deadlines = {j: self.deadlines[j] for j in range(self.num_tasks)}  # Deadline is deadlines[j] for each block

        # Decision Variables
        task_to_block = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="x")  # x[i, j] binary
        task_starting_time = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="t")  # t[i, j] binary
        anti_anxiety = model.addVars(self.num_tasks, vtype=GRB.INTEGER, name="a")

        # Constraint 0 (Temporary): Don't put any tasks on the 0th time block — I have a feeling this is messing up some of our constraints.
        # model.addConstr(sum(task_starting_time[0, j] for j in range(self.num_tasks)) == 0)

        # Constraint 1: Ensure each task is assigned exactly `task_times[j]` blocks in the time horizon
        for j in range(self.num_tasks):
            model.addConstr(
                sum(task_to_block[i, j] for i in range(self.max_time)) == self.task_times[j],
                f"Sum_x_{j}"
            )

        # Constraint 2: Each task must take place in the blocks between its starting time and its dealine
        for i in range(self.max_time):
            model.addConstr(i*task_starting_time[i, j] <= (self.deadlines[j] - self.task_times[j]), f"Bound_t_{i}_{j}")

        # Constraint 2.5: Tasks cannot start on the same block
        for i in range(self.max_time):
            model.addConstr(sum(task_starting_time[i, j] for j in range(self.num_tasks)) <= 1, f"One starting time per time block")

        # Constraint 3: At most 1 task per time block
        for i in range(self.max_time):
            model.addConstr(sum(task_to_block[i, j] for j in range(self.num_tasks)) <= 1, f"One task per time block")

        # Constraint 4: Each task has exactly one starting time
        for j in range(self.num_tasks):
            model.addConstr(
                sum(task_starting_time[i, j] for i in range(self.deadlines[j])) == 1,
                f"Unique_t_{j}"
            )

        # Constraint 5: Linking task_starting_time with task_to_block
        for j in range(self.num_tasks):
            latest_start = min(self.max_time - task_times[j], deadlines[j] - task_times[j])
            for i in range(latest_start + 1):  # Include latest valid start
                for k in range(task_times[j]):
                    model.addConstr(
                        task_to_block[i + k, j] >= task_starting_time[i, j],
                        f"Start_link_{i}_{j}_{k}"
                    )

        # Constraint 6: If a task starts at time t, then it occupies time blocks t to t + d[j] - 1
        model.addConstrs(
            task_to_block[t_prime, j] >= task_starting_time[t, j]
            for j in range(self.num_tasks)
            for t in range(self.max_time - self.deadlines[j] + 1)
            for t_prime in range(t, t + self.task_times[j])
        )
    
        
        # Temporary Objective: Minimize task starting times
        model.setObjective(
            sum(sum(i*task_starting_time[i, j] for j in range(self.num_tasks)) for i in range(self.max_time)),
            GRB.MINIMIZE
        )
        
        
        # Solve the model
        model.optimize()


        task_to_times_array = np.zeros((self.max_time, self.num_tasks), dtype=int)

        for i in range(self.max_time):
            for j in range(self.num_tasks):
                task_to_times_array[i, j] = task_to_block[i, j].X

        start_times_array = np.zeros((self.max_time, self.num_tasks), dtype=int)

        for i in range(self.max_time):
            for j in range(self.num_tasks):
                start_times_array[i, j] = task_starting_time[i, j].X

        print(f"=======task_to_times: ======= \n {task_to_times_array}")
        print(f"=======start_times: ======= \n {start_times_array}")


        # Store results
        results_dict = {}


        if model.status == GRB.OPTIMAL:
            print("Optimal Solution Found:")
            for j in range(self.num_tasks):
                for i in range(self.max_time):
                    if task_starting_time[i, j].x == 1:  # Check if task starts here
                        start_time = i
                        end_time = i + self.task_times[j]
                        results_dict[self.task_names[j]] = (start_time, end_time)
                        print(f"Task {self.task_names[j]} scheduled from {start_time} to {end_time}")

        else:
            print("No optimal solution found.")

        return results_dict


In [43]:
test_suites = dict()

# Instance Test 1
task_times_1 = [1, 1, 2]
deadlines_1 = [30, 30, 30]
task_names_1 = ["task1", "task2", "task3"]
test_suites[1] = [task_times_1, deadlines_1, task_names_1]

# Instance Test 2
task_times_2 = [3, 5, 7, 2]
deadlines_2 = [4, 10, 20, 10]
task_names_2 = ["task1", "task2", "task3", "task4"]
test_suites[2] = [task_times_2, deadlines_2, task_names_2]


# Create instance
cal_1 = PSC_Optimizer(task_times_1, deadlines_1, task_names_1)
cal_2 = PSC_Optimizer(task_times_2, deadlines_2, task_names_2)

# Optimize
print(cal_2.OptimizeCalendar())


Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.2.0 23C71)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2548635 - for non-commercial use only - registered to ba___@rice.edu
Optimize a model with 355 rows, 324 columns and 877 nonzeros
Model fingerprint: 0xce55b897
Variable types: 0 continuous, 324 integer (320 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+00]
Found heuristic solution: objective 33.0000000
Presolve removed 310 rows and 272 columns
Presolve time: 0.00s
Presolved: 45 rows, 52 columns, 194 nonzeros
Found heuristic solution: objective 28.0000000
Variable types: 0 continuous, 52 integer (52 binary)

Root relaxation: objective 1.700000e+01, 36 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Une